In [74]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [75]:
## Load the dataset
data = pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [76]:
### Preprocess the data
## Drop irelevant features
data.drop(['RowNumber','CustomerId','Surname'], axis=1, inplace=True)

In [77]:
### Encode the categorical vaiable
label_encoder_gender = LabelEncoder()
# data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])
data


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [137]:
data.columns

Index(['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts',
       'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited',
       'Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype='str')

In [82]:
### One Hot encode geography column
from sklearn.preprocessing import OneHotEncoder
oneht_encoder_geo = OneHotEncoder()
geo_encoder = oneht_encoder_geo.fit_transform(data[['Geography']])
geo_encoder

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [86]:
geo_encoder.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [81]:
oneht_encoder_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [87]:
geo_encoded_df = pd.DataFrame(geo_encoder.toarray(), columns=oneht_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [89]:
### Combine all the columns with original data
data = pd.concat([data, geo_encoded_df], axis=1)

In [93]:
data.drop(['Geography'], axis=1, inplace=True)

In [94]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [96]:
### Save the encoders and scaler
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(oneht_encoder_geo, file)

In [97]:
### Divide the dataset into independent and dependent dataset
X = data.drop(['Exited'], axis=1)
y = data['Exited']

In [98]:
### Split the data
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)

In [99]:
### Scale down the feature
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [102]:
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

### ANN Implementation

In [103]:
import tensorflow as tf

In [104]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [112]:
X_train.shape[0]

8000

In [113]:
### Build ANN Model
model = Sequential([
Dense(64, activation='relu', input_shape=(X_train.shape[1],)),## HL1
Dense(32,activation='relu'),## HL2
Dense(1,activation='sigmoid')## Output Layer
])

In [114]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [116]:
import tensorflow
opt = tensorflow.keras.optimizers.Adam(learning_rate=0.01)
loss = tensorflow.keras.losses.BinaryCrossentropy()

In [117]:
### Compile the model
# model.compile(optimizer='adam',loss='binary_crossentropy', metrics=['accuracy'])
model.compile(optimizer= opt,loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
## Set up the Tensorboard- to store the model training logs
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [125]:
### Setup Early stoping -> to stop the epochs after certain iterations
early_stoping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [127]:
### Train the model
history = model.fit(
    X_train, y_train, validation_data = (X_test, y_test), epochs=100,
    callbacks = [tensorflow_callback, early_stoping_callback] 
)

Epoch 1/100
250/250 [==============================] - 1s 5ms/step - loss: 0.3125 - accuracy: 0.8709 - val_loss: 0.3413 - val_accuracy: 0.8600
Epoch 2/100
250/250 [==============================] - 1s 5ms/step - loss: 0.3057 - accuracy: 0.8696 - val_loss: 0.3561 - val_accuracy: 0.8585
Epoch 3/100
250/250 [==============================] - 1s 5ms/step - loss: 0.3063 - accuracy: 0.8754 - val_loss: 0.3519 - val_accuracy: 0.8585
Epoch 4/100
250/250 [==============================] - 1s 5ms/step - loss: 0.3037 - accuracy: 0.8746 - val_loss: 0.3469 - val_accuracy: 0.8660
Epoch 5/100
250/250 [==============================] - 1s 4ms/step - loss: 0.3030 - accuracy: 0.8737 - val_loss: 0.3461 - val_accuracy: 0.8650
Epoch 6/100
250/250 [==============================] - 2s 7ms/step - loss: 0.3044 - accuracy: 0.8705 - val_loss: 0.3647 - val_accuracy: 0.8590
Epoch 7/100
250/250 [==============================] - 1s 5ms/step - loss: 0.3011 - accuracy: 0.8741 - val_loss: 0.3596 - val_accuracy: 0.8600

In [128]:
### save the model file
model.save('model.h5')

c:\kaizen\deeplearning\ANN\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [133]:
### Load tensorboard Extention
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [136]:
%tensorboard --logdir logs/fit20260619-134739

Launching TensorBoard...

In [ ]:
### Load the pickle file
